# Module 6 · Solutions

In [ ]:
import pandas as pd, numpy as np
from scipy import stats
import os
BASE = "data/" if os.path.exists("data") else "https://raw.githubusercontent.com/vivekhashtag/financial-analytics-course/main/data/"
fin = pd.read_csv(BASE + "company_financials.csv"); fin["t"] = np.arange(len(fin))
px = pd.read_csv(BASE + "nifty50_prices.csv", parse_dates=["date"]).set_index("date").sort_index()
m = px["close"].resample("ME").last().dropna()

## 6A

In [ ]:
# Ex1 - pre-COVID fit forecasting the COVID year
pre = fin[fin.t <= 3]
r = stats.linregress(pre["t"], pre["revenue_cr"])
fc = r.intercept + r.slope*4
actual = fin.loc[4, "revenue_cr"]
print(f"Forecast FY20-21: {fc:,.0f} cr | Actual: {actual:,.0f} cr | Miss: {fc-actual:+,.0f} cr ({(fc-actual)/actual*100:+.0f}%)")
print("No trend fitted on 2016-19 could foresee 2020. Regime change is what forecasting CANNOT do;")
print("the professional response is intervals + stated regime assumptions, not a cleverer curve.")

# Ex2 - point-in-time refit
r1 = stats.linregress(fin["stores_count"], fin["revenue_cr"])
r2 = stats.linregress(fin["stores_count"], fin["revenue_cr_as_first_reported"])
print(f"\nRestated:      slope {r1.slope:.2f}, R2 {r1.rvalue**2:.4f}")
print(f"First-reported: slope {r2.slope:.2f}, R2 {r2.rvalue**2:.4f}")
print("Tiny shift here (one restated cell) - but you have now IMPLEMENTED point-in-time discipline once.")

# Ex3 - interval from the all-data fit
res_all = stats.linregress(fin["t"], fin["revenue_cr"])
res_post = stats.linregress(fin[fin.t>=6]["t"], fin[fin.t>=6]["revenue_cr"])
s_all  = (fin["revenue_cr"] - (res_all.intercept+res_all.slope*fin["t"])).std(ddof=2)
post = fin[fin.t>=6]
s_post = (post["revenue_cr"] - (res_post.intercept+res_post.slope*post["t"])).std(ddof=2)
print(f"\nInterval half-width: all-data ±{2*s_all:,.0f} cr vs post-COVID ±{2*s_post:,.0f} cr")
print("Present the post-COVID interval WITH the sentence: 'assuming the post-COVID regime holds.'")
print("The all-data band is wider because it silently prices in 'COVID could recur any year' - a different assumption, also worth saying out loud.")

## 6B

In [ ]:
# Ex1 - tuning alpha, and the honesty catch
def ses_forecasts(series, alpha):
    level = series.iloc[0]; fcs = []
    for actual in series:
        fcs.append(level); level = alpha*actual + (1-alpha)*level
    return pd.Series(fcs, index=series.index)

split = "2025-01-31"; test_idx = m[split:][1:].index
for a in [0.2, 0.4, 0.6, 0.8]:
    f = ses_forecasts(m, a).loc[test_idx]
    rmse = np.sqrt(((m.loc[test_idx]-f)**2).mean())
    print(f"alpha {a}: test RMSE {rmse:,.0f}")
print("\nPicking alpha ON the test set contaminates it: the reported score now flatters you.")
print("Clean protocol: choose alpha on train (or a validation slice), touch the test set once, report that.")

In [ ]:
# Ex2 - drift forecaster in the walk-forward
horizon = 24; rec = []
for i in range(len(m)-horizon, len(m)):
    h = m.iloc[:i]
    rec.append({"date": m.index[i], "actual": m.iloc[i],
                "naive": h.iloc[-1],
                "drift": h.iloc[-1] + h.diff().mean()})
wf = pd.DataFrame(rec).set_index("date")
for c_ in ["naive","drift"]:
    e = wf["actual"]-wf[c_]
    print(f"{c_:<6} MAE {e.abs().mean():7,.0f}  RMSE {np.sqrt((e**2).mean()):7,.0f}")
print("\nDrift adds the average up-move and typically edges naive on a trending series - by a sliver.")
print("That sliver IS the trend premium; whether it survives costs and regime shifts is Module 11's question.")